In [1]:
import os
import torch 
import torch.nn as nn
from spnc import spnc_anisotropy
import numpy as np
import matplotlib.pyplot as plt
import tqdm as tqdm
import pickle
import spnc_ml as ml


from pathlib import Path

CANDIDATES = [
    
    Path(r"C:\Users\tom\Desktop\Repository"),
    Path(r"C:\Users\Chen\Desktop\Repository"),
]
searchpaths = [p for p in CANDIDATES if p.exists()]

#tuple of repos
repos = ('machine_learning_library',)

from deterministic_mask import fixed_seed_mask, max_sequences_mask
import repo_tools
repo_tools.repos_path_finder(searchpaths, repos)
from single_node_res import single_node_reservoir
import ridge_regression as RR
from linear_layer import *
from mask import binary_mask
from utility import *
from NARMA10 import NARMA10
from datasets.load_TI46_digits import *
import datasets.load_TI46 as TI46
from sklearn.metrics import classification_report


C:\Users\Chen\Desktop\Repository\machine_learning_library\ridge_regression.py:6: SyntaxWarning: invalid escape sequence '\l'
  '''


In [2]:
# 构建储层对象
class ReservoirParams:
    def __init__(self, **kwargs):
            # Reservoir parameters 
            self.h = 0.4
            self.theta_H = 90
            self.k_s_0 = 0
            self.phi = 45
            self.beta_prime = 35.13826524755751

            # Network parameters 
            self.Nvirt = 50
            self.m0 = 0.005288612874870094
            self.bias = True
            self.Nwarmup = 0
            self.verbose_repr = False

            self.params = {
                'theta': 0.34142235979698393,
                'gamma': 0.069274461903986,
                'delay_feedback': 0,
                'Nvirt': self.Nvirt,
                'length_warmup': self.Nwarmup,
                'warmup_sample': self.Nwarmup * self.Nvirt,
                'voltage_noise': False,
                'seed_voltage_noise': 1234,
                'delta_V': 0.1,
                'johnson_noise': False,
                'seed_johnson_noise': 1234,
                'mean_johnson_noise': 0.0000,
                'std_johnson_noise': 0.00001,
                'thermal_noise': False,
                'seed_thermal_noise': 1234,
                'lambda_ou': 1.0,
                'sigma_ou': 0.1
        }

            for key in ['h', 'theta_H', 'k_s_0', 'phi', 'beta_prime', 'Nvirt', 'm0', 'bias', 'Nwarmup']:
                if key in kwargs:
                    setattr(self, key, kwargs[key])

            
            if 'params' in kwargs and isinstance(kwargs['params'], dict):
                self.params.update(kwargs['params'])

    
    def update_params(self, **kwargs):
        for key, value in kwargs.items():
            if hasattr(self, key):
                setattr(self, key, value)
            if key in self.params:
                self.params[key] = value
            if not hasattr(self, key) and key not in self.params:
                raise AttributeError(f"ReservoirParams has no attribute or param key '{key}'")
            
    def print_params(self, verbose=False):
        if not verbose:
            print(f"ReservoirParams(h={self.h}, beta_prime={self.beta_prime}, Nvirt={self.Nvirt})")
        else:
            print(f"ReservoirParams detailed info:")
            print(f"  h = {self.h}")
            print(f"  theta_H = {self.theta_H}")
            print(f"  k_s_0 = {self.k_s_0}")
            print(f"  phi = {self.phi}")
            print(f"  beta_prime = {self.beta_prime}")
            print(f"  Nvirt = {self.Nvirt}")
            print(f"  m0 = {self.m0}")
            print(f"  bias = {self.bias}")
            print("  params dictionary:")
            for k, v in self.params.items():
                print(f"    {k}: {v}")

params = ReservoirParams(
        h=0.4, m0=0.006937322149792008, Nvirt=200, beta_prime=27.251620432439488,
        params={'theta': 0.01, 'gamma': 0.3663969812988086, 'Nvirt': 200}
    )
# params = ReservoirParams()

spn_bestCQ = spnc_anisotropy(
        params.h,
        params.theta_H,
        params.k_s_0,
        params.phi,
        params.beta_prime,
        restart=True
    )

# transform = spn.gen_signal_slow_delayed_feedback
transform = spn_bestCQ.gen_signal_slow_delayed_feedback

speakers = ['f1','f2'] 

acc = ml.spnc_TI46(speakers, params.Nvirt, params.m0, params.bias, transform, params.params)
print(acc)

Samples for training:  20
Samples for test:  29
Using MFCC preprocessing
Processing signal 1/20
Signal 1 processed with shape: (20, 13)
Processing signal 2/20
Signal 2 processed with shape: (20, 13)
Processing signal 3/20
Signal 3 processed with shape: (20, 13)
Processing signal 4/20
Signal 4 processed with shape: (20, 13)
Processing signal 5/20
Signal 5 processed with shape: (20, 13)
Processing signal 6/20
Signal 6 processed with shape: (20, 13)
Processing signal 7/20
Signal 7 processed with shape: (20, 13)
Processing signal 8/20
Signal 8 processed with shape: (20, 13)
Processing signal 9/20
Signal 9 processed with shape: (20, 13)
Processing signal 10/20
Signal 10 processed with shape: (20, 13)
Processing signal 11/20
Signal 11 processed with shape: (20, 13)
Processing signal 12/20
Signal 12 processed with shape: (20, 13)
Processing signal 13/20
Signal 13 processed with shape: (20, 13)
Processing signal 14/20
Signal 14 processed with shape: (20, 13)
Processing signal 15/20
Signal 15 p

c:\Users\Chen\.conda\envs\spnc_python_3_12\Lib\site-packages\sklearn\metrics\_classification.py:409: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


              precision    recall  f1-score   support

           0      1.000     1.000     1.000        29

    accuracy                          1.000        29
   macro avg      1.000     1.000     1.000        29
weighted avg      1.000     1.000     1.000        29

1.0


c:\Users\Chen\.conda\envs\spnc_python_3_12\Lib\site-packages\sklearn\metrics\_classification.py:409: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
